 # Make Prompt Fine Tuning Dataset

In [ ]:
from __future__ import annotations

import pandas as pd
from datasets import load_dataset

In [ ]:
data_df = pd.read_json(
    "../data/gpt_data.json",
)

In [ ]:
def check_inclusion(row):
    sentence = row["Sentence"]
    for col in ["EventTrigger", "Subject", "Object"]:
        if row[col] and row[col] not in sentence:
            print(
                f"Row {row.name} - {col}:'{row[col]}' - " f"not in :: {row['Sentence']}"
            )


# Apply the function to each row
data_df.apply(check_inclusion, axis=1);

In [ ]:
train_df_list = []
test_df_list = []
eval_df_list = []

for event_type in data_df["EventType"].unique():
    event_data = data_df[data_df["EventType"] == event_type]
    shuffled_data = event_data.sample(frac=1, random_state=42).reset_index(drop=True)
    train_data = shuffled_data.iloc[:24]
    test_data = shuffled_data.iloc[24:34]
    eval_data = shuffled_data.iloc[34:]
    train_df_list.append(train_data)
    test_df_list.append(test_data)
    eval_df_list.append(eval_data)

# Concatenate the lists into DataFrames
train_df = pd.concat(train_df_list).reset_index(drop=True)
test_df = pd.concat(test_df_list).reset_index(drop=True)
eval_df = pd.concat(eval_df_list).reset_index(drop=True)

In [ ]:
train_df.to_csv("../data/gpt_train.tsv", index=False, sep="\t")
test_df.to_csv("../data/gpt_test.tsv", index=False, sep="\t")
eval_df.to_csv("../data/gpt_eval.tsv", index=False, sep="\t")

## Load the dataset

In [ ]:
def format_chat_message(example):
    """This function formats the chat message for the user.

    Args:
        example (dict): A dictionary containing the sentence, event type, event
            trigger, subject, and object.

    Returns:
        dict: A dictionary containing the formatted chat message.
    """
    event_types = (
        "Build",
        "Clean",
        "Cook",
        "Exercise",
        "Hire",
        "Injure",
        "Meet",
        "Purchase",
        "Transport",
        "Travel",
    )
    question = (
        f"I am going to give you a sentence and I want you to tell me what type "
        "of event it is describing and what is the indicating verb, object, "
        "and subject of the sentence. There are ten possible types of events "
        "you can choose from. "
        "Please give your answer in a JSON with the key 'EventType' matching "
        f"one of the following options: {event_types}. The 'EventTrigger' "
        "matching the relevant verb in the sentence. The 'Subject' matching "
        "the actor in the sentence, and 'Object' matching the target object.\n"
        f"The sentence is: {example['Sentence']}"
    )
    answer = (
        f"{{'EventType': '{example['EventType']}', "
        f"'EventTrigger': '{example['EventTrigger']}', "
        f"'Subject': '{example['Subject']}', "
        f"'Object': '{example['Object']}'}}"
    )
    return {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
    }

In [ ]:
dataset = load_dataset(
    "csv",
    data_files={
        "train": "../data/gpt_train.tsv",
        "test": "../data/gpt_test.tsv",
        "eval": "../data/gpt_eval.tsv",
    },
    delimiter="\t",
)

formatted_dataset = dataset.map(
    format_chat_message, remove_columns=dataset["train"].column_names
)

In [ ]:
import textwrap

print(textwrap.fill(str(formatted_dataset["eval"][10]["messages"]), 80))